In [ ]:
!pip install -q torch transformers datasets matplotlib numpy

In [ ]:
import gc
import re
import time
import warnings
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import AutoTokenizer, DynamicCache, Qwen3ForCausalLM

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
MODEL_NAME = "dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1"
BLOCK_MODEL_NAME = "dllm-hub/Qwen3-0.6B-diffusion-bd3lm-v0.1"
AR_MODEL_NAME = "Qwen/Qwen3-0.6B"
DTYPE = torch.bfloat16
GEN_LEN = 256
BLOCK_SIZE = 32
NUM_QUESTIONS = 10
SEED = 0
torch.manual_seed(SEED)
print(f"DEVICE = {DEVICE}")

# Diffusion Language Models

An autoregressive (AR) model writes one token per forward pass, left to right, and never revises
a token. A **masked diffusion language model** starts from a reply made entirely of `[MASK]`
tokens and fills in many of them per forward pass, in whatever order it is most confident about.
This notebook builds the training objective, the sampler, block diffusion with an exact KV cache
and confidence-threshold parallel decoding from scratch on two real 0.6B diffusion checkpoints,
and measures them against the AR model they were adapted from on GSM8K.

---

# 1) Introduction

## 1.1 The problem

Decoding cost of an AR model is one full forward pass per generated token. At small batch sizes
this is memory-bandwidth bound: the GPU streams all weights to produce a single token. Producing
$k$ tokens per pass would cut latency roughly $k$-fold, but a left-to-right factorisation
$p(x) = \prod_i p(x_i \mid x_{<i})$ has no slot for that.

## 1.2 Where earlier approaches fall short

| Approach | Limitation |
|---|---|
| Speculative decoding | Still AR; speed-up capped by the draft model's acceptance rate |
| Multi-token prediction heads | Extra heads guess a few tokens ahead, then must be verified |
| Non-autoregressive translation (2018–20) | One-shot parallel output; tokens disagree with each other ("multimodality problem") |
| Continuous diffusion on embeddings | Rounding back to discrete tokens is lossy; never matched AR at scale |

## 1.3 The core idea

Treat `[MASK]` as an absorbing noise state. Training corrupts a random fraction $t$ of the tokens
to `[MASK]` and asks a **bidirectional** transformer to recover them. Sampling runs the corruption
backwards: start fully masked, predict every masked position at once, keep the confident
predictions, repeat. The number of passes is a free knob between 1 and the sequence length.

```
step 0   ▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒       every position masked
step 1   ▒▒▒▒▒ is ▒▒▒▒▒▒▒▒▒▒▒▒ 84 .     most confident tokens first, any position
step 2   12 × 7 is ▒▒ because ▒▒ 84 .
step 3   12 × 7 is 84 because … 84 .    several tokens committed per forward pass
```

## 1.4 Findings from the past two years

| Model / work | Date | What is new |
|---|---|---|
| MDLM (Sahoo et al.) | 2024 | Simple masked-diffusion ELBO: a weighted masked-LM loss |
| LLaDA 8B | Feb 2025 | First masked dLLM trained from scratch at 8B, competitive with LLaMA-3 8B |
| Mercury (Inception) / Gemini Diffusion | mid 2025 | Commercial dLLMs reporting >1,000 tokens/s on coding tasks |
| Dream 7B | 2025 | dLLM initialised from an AR model (Qwen2.5) instead of from scratch |
| BD3-LM (Arriola et al.) | ICLR 2025 | Block diffusion: AR across blocks, diffusion within; exact KV cache, any length |
| Fast-dLLM | 2025 | Training-free speed-ups: approximate KV cache + confidence-threshold parallel decoding |
| Seed Diffusion, LLaDA 2.x, MMaDA / LLaDA-V | 2025 | Scaled and multimodal dLLMs |
| dLLM library (Zhou et al.) | Feb 2026 | Recipes that turn small AR models into MDLM / BD3-LM; the checkpoints used here |

## 1.5 What this notebook covers

| Section | Content |
|---|---|
| 2) Setup | Qwen3-0.6B adapted to MDLM and to BD3-LM, the AR Qwen3-0.6B, GSM8K |
| 3) The math | Absorbing forward process, the ELBO and its $1/t$ weight, reverse sampling, block diffusion |
| 4) From scratch | Bidirectional attention; the MDLM loss measured on real answers; confidence-ranked sampler; decoding order; staircase mask with an exact KV cache |
| 5) End-to-end | GSM8K accuracy, forward passes and wall time: AR vs MDLM vs BD3-LM |
| 6) Variants | Tokens per step, confidence-threshold decoding, infilling |
| 7) Practical notes | Where dLLMs win today, failure modes, serving |
| 8) Conclusion | Decision table |

---

# 2) Setup

## 2.1 Models

All three models share Qwen3-0.6B's architecture, weights format and tokenizer:

| Constant | Checkpoint | Attention at inference |
|---|---|---|
| `MODEL_NAME` | Qwen3-0.6B fine-tuned with the MDLM objective (SFT data: Tulu 3, SmolTalk, OpenCoder) | Full bidirectional |
| `BLOCK_MODEL_NAME` | Same recipe with the BD3-LM objective, block size 32 | Block-causal ("staircase") |
| `AR_MODEL_NAME` | Qwen3-0.6B, the AR model the two were adapted from | Causal |

The diffusion checkpoints ship custom modelling code whose only change to Qwen3 is the attention
mask. We therefore load them into the stock `Qwen3ForCausalLM` and pass the mask ourselves as a
4D boolean tensor, which `transformers` uses as is. The tokenizer adds `<|mask|>` as the
`[MASK]` token.

In [ ]:
def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


def free_memory() -> None:
    """Release cached accelerator memory between experiments."""
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def load(name: str) -> Qwen3ForCausalLM:
    """Load a Qwen3-architecture checkpoint in DTYPE on DEVICE, in eval mode."""
    return Qwen3ForCausalLM.from_pretrained(name, dtype=DTYPE).to(DEVICE).eval()


start = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = load(MODEL_NAME)
block_model = load(BLOCK_MODEL_NAME)
ar_model = load(AR_MODEL_NAME)
MASK_ID = tokenizer.mask_token_id
EOS_IDS = torch.tensor(
    [tokenizer.convert_tokens_to_ids(t) for t in ["<|im_end|>", "<|endoftext|>"]],
    device=DEVICE,
)
print(f"[MASK] = {tokenizer.mask_token!r} (id {MASK_ID}) | EOS ids {EOS_IDS.tolist()}")
print(
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} M params each | "
    f"load {time.time() - start:.0f} s | device memory {device_memory_gb():.2f} GB"
)

## 2.2 Data: GSM8K

Grade-school maths word problems with a worked solution ending in `#### <answer>`. Training-split
solutions are used to measure the loss; test-split questions are used for generation. The
solutions carry calculator annotations `<<48/2=24>>`, which are stripped.

In [ ]:
gsm_train = load_dataset("openai/gsm8k", "main", split="train").select(range(8))
gsm_test = load_dataset("openai/gsm8k", "main", split="test").select(
    range(NUM_QUESTIONS)
)


def chat_prompt(question: str) -> str:
    """User turn plus the assistant header, thinking disabled."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def clean_solution(answer: str) -> str:
    """Remove GSM8K calculator annotations such as <<48/2=24>>."""
    return re.sub(r"<<.*?>>", "", answer)


print(chat_prompt(gsm_train[0]["question"]) + clean_solution(gsm_train[0]["answer"]))

---

# 3) The math

## 3.1 Forward process: absorbing masks

Let $x_0 = (x_0^1, \dots, x_0^L)$ be a token sequence of length $L$ and $t \in [0, 1]$ a noise
level. Each token is replaced by `[MASK]` (written $\mathrm{M}$) independently with probability
$t$:

$$
q(x_t^i \mid x_0^i) = (1 - t)\,\mathbf{1}[x_t^i = x_0^i] + t\,\mathbf{1}[x_t^i = \mathrm{M}].
$$

At $t = 0$ the sequence is clean, at $t = 1$ it is all masks. A masked token never changes back
on its own, so `[MASK]` is an *absorbing* state, and the only information in $x_t$ about a masked
position comes from the unmasked tokens around it.

## 3.2 Reverse process and the model

For $s < t$, the true reverse step keeps each masked token masked with probability $s/t$ and
otherwise reveals it as its clean value $x_0^i$:

$$
q(x_s^i \mid x_t^i = \mathrm{M}, x_0) = \tfrac{s}{t}\,\mathbf{1}[x_s^i = \mathrm{M}]
+ \big(1 - \tfrac{s}{t}\big)\,\mathbf{1}[x_s^i = x_0^i],
\qquad
q(x_s^i \mid x_t^i \ne \mathrm{M}) = \mathbf{1}[x_s^i = x_t^i].
$$

The only unknown is $x_0^i$, so the network is a denoiser $p_\theta(x_0^i \mid x_t)$ that
outputs a distribution over the vocabulary at every masked position, all in one pass. Nothing
asks position $i$ to ignore position $j > i$, so the transformer uses **full bidirectional
attention**. It needs no time input: the fraction of masks in $x_t$ reveals $t$.

## 3.3 Training objective: a weighted masked-LM loss

The continuous-time ELBO (MDLM, LLaDA) bounds the negative log-likelihood:

$$
-\log p_\theta(x_0) \;\le\; \mathcal{L}(\theta) =
\mathbb{E}_{t \sim \mathcal{U}(0,1)}\; \mathbb{E}_{x_t \sim q(\cdot \mid x_0)}
\left[\frac{1}{t} \sum_{i=1}^{L} \mathbf{1}[x_t^i = \mathrm{M}]\,
\big(-\log p_\theta(x_0^i \mid x_t)\big)\right].
$$

| Symbol | Meaning |
|---|---|
| $\mathbf{1}[x_t^i = \mathrm{M}]$ | Loss only on masked positions, as in BERT |
| $1/t$ | About $tL$ positions are masked, so dividing by $t$ turns the sum into a per-sequence quantity independent of how many were masked |
| $\mathbb{E}_t$ | Every masking rate from 0 to 100%, unlike BERT's fixed 15% |

The $1/t$ weight comes from the general masked-diffusion ELBO weight $\alpha_t' / (1 - \alpha_t)$
with keep-probability $\alpha_t = 1 - t$. The bound is tight when, for every masking rate, the
model is a perfect conditional predictor. Integrating over $t$ is equivalent to averaging an
*any-order* autoregressive NLL over random token orders. AR fixes one order, so a dLLM pays some
likelihood for not knowing the order in advance.

For supervised fine-tuning only the reply is masked; the prompt stays clean and conditions every
prediction. Dividing $\mathcal{L}$ by the reply length gives a per-token upper bound comparable to
an AR model's per-token NLL.

## 3.4 Sampling

Start from $x_1$: the prompt followed by $L$ masks. With $S$ steps, each step predicts
$\hat x_0^i = \arg\max p_\theta(\cdot \mid x_t)$ at every masked position and commits $k = L/S$
of them. Sampling the exact reverse process would choose positions at random. **Low-confidence
remasking** (LLaDA) commits the $k$ positions with the highest $\max_v p_\theta(v \mid x_t)$ and
leaves the rest masked. The forward-pass count is $S$, not $L$, and $L/S$ tokens are produced per
pass.

Committing $k$ tokens in one pass samples each from its own marginal. If two positions are
dependent ("New York" vs "San Francisco"), independent marginal choices can disagree ("New
Francisco"). This is why quality drops as $k$ grows, and why confidence matters: a position
whose marginal is near one-hot has little left to disagree about.

## 3.5 Block diffusion

Full-sequence diffusion needs $L$ fixed in advance, and because every position can change, no
key/value can be cached. **Block diffusion** factorises over blocks $x^{(1)}, \dots, x^{(B)}$ of
size $b$:

$$
p_\theta(x) = \prod_{j=1}^{B} p_\theta\big(x^{(j)} \mid x^{(<j)}\big),
$$

with each factor a masked diffusion over $b$ tokens. A token in block $j$ attends to all of
blocks $\le j$ and never to later ones, which gives the block-causal ("staircase") mask:

```
          keys: block 1   block 2   block 3
query block 1   ███       ·         ·
query block 2   ███       ███       ·
query block 3   ███       ███       ███
```

A finished block's keys and values depend only on earlier blocks, so they can be **cached
exactly**, and generation stops at the first block that emits end-of-sequence. With $b = L$
this is MDLM, and with $b = 1$ it is an AR model.

---

# 4) From-scratch implementation

## 4.1 Bidirectional attention

The whole architectural difference from an AR model is one mask. Below, a prompt is followed by
16 masks and the MDLM checkpoint fills all of them in one pass, first with the full mask it was
trained with, then with the default causal mask, under which a masked position cannot see the
answer's context to its right.

In [ ]:
def full_mask(seq_len: int) -> torch.Tensor:
    """(1, 1, seq_len, seq_len) boolean mask: every token attends to every token."""
    return torch.ones(1, 1, seq_len, seq_len, dtype=torch.bool, device=DEVICE)


def masked_canvas(prompt: str, num_masks: int) -> torch.Tensor:
    """Prompt token ids followed by num_masks [MASK] ids -> (1, seq_len)."""
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    masks = torch.full((1, num_masks), MASK_ID, device=DEVICE)
    return torch.cat([prompt_ids, masks], dim=1)


x = masked_canvas(chat_prompt("What is 12 times 7?"), 16)
prompt_len = x.shape[1] - 16
with torch.no_grad():
    logits_full = model(x, attention_mask=full_mask(x.shape[1])).logits
    logits_causal = model(x).logits
for name, logits in [("bidirectional", logits_full), ("causal", logits_causal)]:
    print(f"{name:13s}: {tokenizer.decode(logits[0, prompt_len:].argmax(-1))!r}")

One pass gets the content right but not a fluent sentence: the 16 positions were predicted
independently from their own marginals, so neighbouring positions repeat each other.
Iterative unmasking fixes this.

The same mask is why an AR KV cache cannot be reused. Below, the last masked token is replaced
by a real token. Under bidirectional attention the hidden state of the *first* prompt token
changes, so every cached key and value would be stale.

In [ ]:
x_edit = x.clone()
# Qwen tokenizes numbers digit by digit, so "84" starts with the token "8"
x_edit[0, -1] = tokenizer("84").input_ids[0]
with torch.no_grad():
    for name, mask in [("bidirectional", full_mask(x.shape[1])), ("causal", None)]:
        hidden = [
            model(ids, attention_mask=mask, output_hidden_states=True).hidden_states
            for ids in (x, x_edit)
        ]
        change = (hidden[0][-1][0, 0] - hidden[1][-1][0, 0]).abs().max()
        print(f"{name:13s}: first-token hidden state changes by {change:.3f}")

## 4.2 The MDLM loss on real answers

`mask_tokens` is the absorbing forward process restricted to the reply; `mdlm_loss` is the
$1/t$-weighted masked cross-entropy divided by the reply length. Each GSM8K solution is replicated `num_t` times
with stratified noise levels $t_k = (k + u)/K$, which covers $[0, 1]$ evenly and lowers the
variance of the Monte Carlo estimate.

In [ ]:
def mask_tokens(
    input_ids: torch.Tensor, maskable: torch.Tensor, t: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """input_ids, maskable (batch_num, seq_len), t (batch_num,) -> (noisy, masked)."""
    masked = (torch.rand(input_ids.shape, device=DEVICE) < t[:, None]) & maskable
    return input_ids.masked_fill(masked, MASK_ID), masked


def mdlm_loss(
    model: Qwen3ForCausalLM,
    input_ids: torch.Tensor,
    maskable: torch.Tensor,
    t: torch.Tensor,
    bidirectional: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    """input_ids, maskable (batch_num, seq_len), t (batch_num,).

    Returns the per-token NLL bound (batch_num,) and the cross-entropy at every
    reply position (batch_num, reply_len), zero where nothing was masked.
    """
    noisy, masked = mask_tokens(input_ids, maskable, t)
    mask = full_mask(input_ids.shape[1]) if bidirectional else None
    # the LM head only runs on reply positions: full-vocabulary logits for the
    # whole batch would take several GB
    reply = maskable[0].nonzero().flatten()
    logits = model(noisy, attention_mask=mask, logits_to_keep=reply).logits
    ce = F.cross_entropy(
        logits.float().transpose(1, 2), input_ids[:, reply], reduction="none"
    )
    ce = ce * masked[:, reply]
    return ce.sum(1) / t / maskable.sum(1), ce


def encode_example(question: str, solution: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Chat-formatted (prompt + reply) ids (1, seq_len) and a reply-only mask."""
    prompt_ids = tokenizer(chat_prompt(question)).input_ids
    reply_ids = tokenizer(clean_solution(solution) + "<|im_end|>").input_ids
    input_ids = torch.tensor([prompt_ids + reply_ids], device=DEVICE)
    maskable = torch.zeros_like(input_ids, dtype=torch.bool)
    maskable[0, len(prompt_ids) :] = True
    return input_ids, maskable


input_ids, maskable = encode_example(gsm_train[0]["question"], gsm_train[0]["answer"])
t = torch.tensor([0.25, 0.5, 0.9], device=DEVICE)
with torch.no_grad():
    bound, ce = mdlm_loss(model, input_ids.repeat(3, 1), maskable.repeat(3, 1), t)
print(f"{maskable.sum().item()} reply tokens")
for k in range(3):
    num_masked = (ce[k] > 0).sum().item()
    print(f"t = {t[k]:.1f}: {num_masked:3d} masked | per-token bound {bound[k]:.3f}")

Now the estimate over 8 solutions with 8 noise levels each, for three models on the same
tokens:

* the MDLM checkpoint with its bidirectional mask: a valid upper bound on its NLL;
* the same weights forced to be causal: the objective it was not trained for;
* the AR Qwen3-0.6B: its exact per-token NLL of the reply, by teacher forcing.

In [ ]:
NUM_T = 8
torch.manual_seed(SEED)
bounds = {"MDLM, bidirectional": [], "MDLM, causal mask": []}
ar_nll, ce_by_t = [], []
start = time.time()
with torch.no_grad():
    for row in gsm_train:
        input_ids, maskable = encode_example(row["question"], row["answer"])
        t = (
            torch.arange(NUM_T, device=DEVICE) + torch.rand(NUM_T, device=DEVICE)
        ) / NUM_T
        ids, keep = input_ids.repeat(NUM_T, 1), maskable.repeat(NUM_T, 1)
        for name, bidirectional in [
            ("MDLM, bidirectional", True),
            ("MDLM, causal mask", False),
        ]:
            bound, ce = mdlm_loss(model, ids, keep, t, bidirectional)
            bounds[name].append(bound.mean().item())
            if bidirectional:
                num_masked = (ce > 0).sum(1).clamp(min=1)
                ce_by_t += list(zip(t.tolist(), (ce.sum(1) / num_masked).tolist()))
        # AR: logits at position i predict token i + 1
        reply = maskable[0].nonzero().flatten()
        logits = ar_model(input_ids, logits_to_keep=reply - 1).logits[0].float()
        ar_nll.append(F.cross_entropy(logits, input_ids[0, reply]).item())
free_memory()
for name, values in bounds.items():
    print(f"{name:22s}: per-token NLL bound {np.mean(values):.3f}")
print(f"{'AR Qwen3-0.6B':22s}: per-token NLL       {np.mean(ar_nll):.3f}")
print(f"{time.time() - start:.0f} s")

The bound for the bidirectional model is the loss MDLM training minimises. Forcing the same
weights to be causal makes it about four times worse, so the adaptation really moved the model
to bidirectional attention.

The AR number is not a like-for-like comparison. The MDLM checkpoint was fine-tuned on SFT data
that includes step-by-step maths solutions, while Qwen3-0.6B was post-trained for chat and
thinking traces, not for GSM8K's terse `####` format. Against the same training data, theory says
the AR model should win: the dLLM's bound averages over every unmasking order, and AR fixes the
one order that suits language. Since the bound is an upper bound, MDLM's true NLL is lower still.

The per-masked-token cross-entropy rises with $t$. With few masks, a masked token is surrounded
by its true context; with many, the model must guess the reply from the prompt alone.

In [ ]:
t_values, ce_values = np.array(ce_by_t).T
bins = np.linspace(0, 1, 6)
centres = (bins[:-1] + bins[1:]) / 2
means = [
    ce_values[(t_values >= lo) & (t_values < hi)].mean()
    for lo, hi in zip(bins[:-1], bins[1:])
]
plt.figure(figsize=(5, 3))
plt.scatter(t_values, ce_values, s=8, alpha=0.4, label="one (solution, t) sample")
plt.plot(centres, means, "o-", color="black", label="bin mean")
plt.xlabel("masking rate t")
plt.ylabel("cross-entropy per masked token")
plt.legend()
plt.tight_layout()
plt.show()

## 4.3 Committing tokens

One step of the sampler, shared by everything that follows. Given the logits of the current
block, `commit_tokens` picks which masked positions to reveal:

| Rule | Positions committed |
|---|---|
| Fixed count, low-confidence remasking (LLaDA) | The `tokens_per_step` masked positions with the highest $\max_v p(v)$ |
| Confidence threshold (Fast-dLLM) | Every masked position with $\max_v p(v) > \tau$, at least one |

Decoding is greedy (argmax) throughout; adding Gumbel noise to the logits would give sampling.

In [ ]:
def commit_tokens(
    logits: torch.Tensor,
    block: torch.Tensor,
    tokens_per_step: int = 1,
    threshold: float | None = None,
) -> torch.Tensor:
    """logits (block_len, vocab_size), block (block_len,) -> updated block."""
    masked = block == MASK_ID
    confidence, prediction = logits.float().softmax(-1).max(-1)
    confidence = confidence.masked_fill(~masked, -1.0)
    if threshold is not None:
        k = max(1, int((confidence > threshold).sum()))
    else:
        k = tokens_per_step
    commit = confidence.topk(min(k, int(masked.sum()))).indices
    block = block.clone()
    block[commit] = prediction[commit]
    return block


block = x[0, prompt_len:]
logits = logits_full[0, prompt_len:]
for kwargs in [{"tokens_per_step": 1}, {"tokens_per_step": 4}, {"threshold": 0.9}]:
    new_block = commit_tokens(logits, block, **kwargs)
    committed = (new_block != MASK_ID).nonzero().flatten().tolist()
    print(
        f"{str(kwargs):24s} -> positions {committed}: "
        f"{tokenizer.decode(new_block[committed])!r}"
    )

## 4.4 The MDLM sampler

`diffusion_generate` lays out the prompt followed by `GEN_LEN` masks and walks over the reply in
blocks of `block_size`. Inside a block it runs the full sequence through the model, commits
tokens, and repeats until the block has no masks. Later blocks stay masked but visible, as in
training. Once a finished block contains an end-of-sequence token, the remaining blocks are
skipped. With `block_size = GEN_LEN` this is the pure full-sequence sampler.

`logits_to_keep` restricts the LM head to the current block's positions, which avoids a
$(\text{seq\_len} \times 151{,}936)$ logit tensor per step.

In [ ]:
@torch.no_grad()
def diffusion_generate(
    model: Qwen3ForCausalLM,
    question: str,
    block_size: int = BLOCK_SIZE,
    gen_len: int = GEN_LEN,
    **commit_kwargs,
) -> dict:
    """Masked-diffusion decoding of a reply to question.

    Returns text, nfe (forward passes) and commit_step (gen_len,): the forward pass at
    which each reply position was committed, -1 if never.
    """
    x = masked_canvas(chat_prompt(question), gen_len)
    prompt_len = x.shape[1] - gen_len
    attention_mask = full_mask(x.shape[1])
    commit_step = torch.full((gen_len,), -1)
    nfe = 0
    for block_start in range(prompt_len, x.shape[1], block_size):
        block_end = min(block_start + block_size, x.shape[1])
        while (x[0, block_start:block_end] == MASK_ID).any():
            positions = torch.arange(block_start, block_end, device=DEVICE)
            logits = model(
                x, attention_mask=attention_mask, logits_to_keep=positions
            ).logits[0]
            before = x[0, block_start:block_end]
            after = commit_tokens(logits, before, **commit_kwargs)
            nfe += 1
            changed = (after != before).nonzero().flatten().cpu()
            commit_step[block_start - prompt_len + changed] = nfe
            x[0, block_start:block_end] = after
        if torch.isin(x[0, block_start:block_end], EOS_IDS).any():
            break
    return {"ids": x[0, prompt_len:], "nfe": nfe, "commit_step": commit_step}

The returned ids still hold masks after the stopping block and padding after the end of
sequence. `reply_text` cuts at the first end-of-sequence token.

In [ ]:
def reply_text(ids: torch.Tensor) -> tuple[str, int]:
    """Reply ids -> (text up to the first EOS or mask, number of tokens kept)."""
    stop = torch.isin(ids, EOS_IDS) | (ids == MASK_ID)
    length = int(stop.nonzero()[0]) if stop.any() else len(ids)
    return tokenizer.decode(ids[:length]), length


question = gsm_test[0]["question"]
start = time.time()
out = diffusion_generate(model, question)
text, length = reply_text(out["ids"])
print(f"{question}\n\n{text}\n")
print(
    f"{length} tokens in {out['nfe']} forward passes | {time.time() - start:.1f} s | "
    f"reference: {gsm_test[0]['answer'].split('####')[-1].strip()}"
)

## 4.5 Decoding order

`commit_step` records when each position was filled. Compare one block of 256 (pure diffusion)
with blocks of 32. In the pure run, the model is most confident about the end of the canvas,
which must be end-of-sequence padding after a short reply, and fills it first. The text itself
appears out of order. With blocks, the order is forced into a left-to-right staircase, and
decoding stops after the block that ends the reply.

In [ ]:
runs = {
    "one block of 256": diffusion_generate(model, question, block_size=GEN_LEN),
    "blocks of 32": diffusion_generate(model, question, block_size=BLOCK_SIZE),
}
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (name, run) in zip(axes, runs.items()):
    done = run["commit_step"] >= 0
    ax.scatter(torch.arange(GEN_LEN)[done], run["commit_step"][done], s=4)
    ax.set_title(f"{name}: {run['nfe']} forward passes")
    ax.set_xlabel("reply position")
axes[0].set_ylabel("forward pass that committed it")
plt.tight_layout()
plt.show()

steps = runs["one block of 256"]["commit_step"]
ids = runs["one block of 256"]["ids"].cpu()
for step in [16, 64, 128]:
    shown = torch.where((steps > 0) & (steps <= step), ids, MASK_ID)
    snapshot = tokenizer.decode(shown[:80]).replace(tokenizer.mask_token, "▒")
    print(f"after {step:3d} passes: {snapshot!r}\n")

## 4.6 Block diffusion with an exact KV cache

The BD3-LM checkpoint was trained with the staircase mask, blocks aligned to absolute
positions ($\text{block} = \lfloor \text{position} / b \rfloor$). The prompt usually ends inside a
block, and its last tokens share that block with the first reply tokens. Their keys therefore
depend on the reply. So the cache holds only *complete* blocks, and the prompt's trailing partial
block is re-run with the first masks.

```
positions  0 ……… 31 | 32 ……… 63 | 64 … 77  78 ……… 95 | 96 ……… 127
           prompt     prompt     | prompt     [MASK]…  | [MASK]…
           └──── cached ────────┘ └─ current block ───┘   next block
```

Each denoising step runs only the current block's $b$ queries against the cached keys plus the
block's own. After the step, `DynamicCache.crop` drops the block's keys again. Once the block is
clean, one more pass appends its keys for good.

In [ ]:
def staircase_mask(seq_len: int, block_size: int = BLOCK_SIZE) -> torch.Tensor:
    """(1, 1, seq_len, seq_len) block-causal mask: block j attends to blocks <= j."""
    block_ids = torch.arange(seq_len, device=DEVICE) // block_size
    return (block_ids[None, :] <= block_ids[:, None])[None, None]


print(staircase_mask(8, block_size=3)[0, 0].int())

First the cache is checked against a single full pass with the staircase mask over the whole
sequence: the logits of the current block must match. The prompt is a full GSM8K question, so
it spans several blocks.

In [ ]:
def prefill(
    model: Qwen3ForCausalLM, prompt_ids: torch.Tensor, block_size: int = BLOCK_SIZE
) -> tuple[DynamicCache, torch.Tensor]:
    """Cache the complete blocks of prompt_ids (1, prompt_len).

    Returns the cache and the prompt tokens of the partial last block (num_leftover,).
    """
    cached_len = prompt_ids.shape[1] // block_size * block_size
    cache = DynamicCache(config=model.config)
    if cached_len > 0:
        model(
            prompt_ids[:, :cached_len],
            attention_mask=staircase_mask(cached_len, block_size),
            past_key_values=cache,
            use_cache=True,
        )
    return cache, prompt_ids[0, cached_len:]


def block_forward(
    model: Qwen3ForCausalLM, block: torch.Tensor, cache: DynamicCache
) -> torch.Tensor:
    """Run block (block_len,) after the cached prefix -> logits (block_len, vocab_size).

    The block's own keys are appended to the cache; the caller crops them if needed.
    """
    cached_len = cache.get_seq_length()
    total_len = cached_len + len(block)
    mask = torch.ones(1, 1, len(block), total_len, dtype=torch.bool, device=DEVICE)
    positions = torch.arange(cached_len, total_len, device=DEVICE)[None]
    out = model(
        block[None],
        attention_mask=mask,
        position_ids=positions,
        past_key_values=cache,
        use_cache=True,
    )
    return out.logits[0]

In [ ]:
prompt_ids = tokenizer(chat_prompt(question), return_tensors="pt").input_ids.to(DEVICE)
cache, leftover = prefill(block_model, prompt_ids)
num_masks = BLOCK_SIZE - len(leftover)
block = torch.cat([leftover, torch.full((num_masks,), MASK_ID, device=DEVICE)])
x = torch.cat([prompt_ids[0], block[len(leftover) :]])[None]
with torch.no_grad():
    cached_logits = block_forward(block_model, block, cache).float()
    reference = (
        block_model(x, attention_mask=staircase_mask(x.shape[1])).logits[0].float()
    )
reference = reference[-BLOCK_SIZE:]
print(
    f"prompt {prompt_ids.shape[1]} tokens: "
    f"{cache.get_seq_length()} cached, {len(leftover)} leftover"
)
print(
    f"max |cached - full| logit = {(cached_logits - reference).abs().max():.3f} "
    f"(logit scale {reference.abs().max():.0f}, {DTYPE}) | "
    "argmax agreement "
    f"{(cached_logits.argmax(-1) == reference.argmax(-1)).float().mean():.0%}"
)

The cached and full passes agree up to bfloat16 rounding. The sampler reuses `commit_tokens`
unchanged; only the forward pass differs from the MDLM sampler.

In [ ]:
@torch.no_grad()
def block_diffusion_generate(
    model: Qwen3ForCausalLM,
    question: str,
    block_size: int = BLOCK_SIZE,
    gen_len: int = GEN_LEN,
    **commit_kwargs,
) -> dict:
    """BD3-LM decoding with an exact prefix KV cache; returns ids and nfe."""
    prompt_ids = tokenizer(chat_prompt(question), return_tensors="pt").input_ids
    cache, leftover = prefill(model, prompt_ids.to(DEVICE), block_size)
    nfe, reply = 1, []
    while sum(len(b) for b in reply) < gen_len:
        cached_len = cache.get_seq_length()
        num_masks = block_size - len(leftover)
        block = torch.cat([leftover, torch.full((num_masks,), MASK_ID, device=DEVICE)])
        while (block == MASK_ID).any():
            logits = block_forward(model, block, cache)
            cache.crop(cached_len)
            block = commit_tokens(logits, block, **commit_kwargs)
            nfe += 1
        # the clean block becomes part of the cached prefix
        block_forward(model, block, cache)
        nfe += 1
        reply.append(block[len(leftover) :])
        leftover = block[:0]
        if torch.isin(reply[-1], EOS_IDS).any():
            break
    return {"ids": torch.cat(reply), "nfe": nfe}


start = time.time()
out = block_diffusion_generate(block_model, question)
text, length = reply_text(out["ids"])
print(f"{text}\n")
print(f"{length} tokens in {out['nfe']} forward passes | {time.time() - start:.1f} s")

---

# 5) End-to-end: GSM8K

Ten test questions, greedy decoding, at most 256 reply tokens, one token committed per step.
The answer is the content of the last `\boxed{}` or, failing that, the last number. The AR
baseline uses `generate` with its KV cache. "Tokens/pass" is reply tokens divided by forward
passes, the quantity diffusion decoding is meant to raise.

In [ ]:
def extract_answer(text: str) -> str | None:
    """Last \\boxed{...} content, else the last number; commas removed."""
    boxed = re.findall(r"\\boxed\{([^{}]*)\}", text)
    candidates = boxed or re.findall(r"-?\d[\d,]*\.?\d*", text)
    if not candidates:
        return None
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", candidates[-1])
    return numbers[-1].replace(",", "").rstrip(".") if numbers else None


@torch.no_grad()
def ar_generate(model: Qwen3ForCausalLM, question: str) -> dict:
    """Greedy AR decoding with a KV cache; one forward pass per new token."""
    prompt_ids = tokenizer(chat_prompt(question), return_tensors="pt").input_ids
    out = model.generate(
        prompt_ids.to(DEVICE),
        max_new_tokens=GEN_LEN,
        do_sample=False,
        top_p=None,
        top_k=None,
        temperature=None,
    )
    ids = out[0, prompt_ids.shape[1] :]
    return {"ids": ids, "nfe": len(ids)}


print(extract_answer("so 3 + 4 = 7 and the total is \\boxed{1,234}."))
print(extract_answer("The answer is 18 dollars."))

`evaluate` runs one decoder over all questions and collects accuracy, forward passes, reply length
and wall time.

In [ ]:
def evaluate(name: str, generate: Callable[[str], dict]) -> dict:
    """Run generate(question) on every test question and summarise."""
    correct, nfe, tokens = 0, 0, 0
    start = time.time()
    for row in gsm_test:
        out = generate(row["question"])
        text, length = reply_text(out["ids"])
        gold = row["answer"].split("####")[-1].strip().replace(",", "")
        correct += extract_answer(text) == gold
        nfe += out["nfe"]
        tokens += length
    seconds = time.time() - start
    free_memory()
    return {
        "method": name,
        "accuracy": correct / len(gsm_test),
        "nfe": nfe / len(gsm_test),
        "tokens": tokens / len(gsm_test),
        "tokens_per_pass": tokens / nfe,
        "seconds": seconds / len(gsm_test),
    }


def print_results(results: list[dict]) -> None:
    """Print a table of evaluate() summaries."""
    print(f"{'method':34s} | acc  | passes | tokens | tokens/pass | s/answer")
    for r in results:
        print(
            f"{r['method']:34s} | {r['accuracy']:.2f} "
            f"| {r['nfe']:6.0f} | {r['tokens']:6.0f} | "
            f"{r['tokens_per_pass']:11.2f} | {r['seconds']:8.1f}"
        )


results = [
    evaluate("AR Qwen3-0.6B", lambda q: ar_generate(ar_model, q)),
    evaluate("MDLM, blocks of 32", lambda q: diffusion_generate(model, q)),
    evaluate("BD3-LM + KV cache", lambda q: block_diffusion_generate(block_model, q)),
]
print_results(results)
print(f"device memory {device_memory_gb():.2f} GB")

At one token per step, the diffusion models need about as many passes as the AR model, and each
pass costs more. MDLM re-reads the whole 256-token canvas every step, which makes it roughly
8× slower per answer. BD3-LM re-reads only one block of 32 against its cache and stays close to
AR speed.

Ten questions cannot rank the models: one answer is 10 points, and the 95% interval at this size
is about ±30 points. The published GSM8K scores for these checkpoints, on the full test set, are
Qwen3-0.6B-Base 59.6, BD3-LM 46.6 and MDLM 29.8. Adapting an AR model to diffusion costs quality
at this scale, and less for BD3-LM, whose block-causal structure stays closer to the AR model it
started from. The point of diffusion decoding is the next section: committing several tokens per
pass.

---

# 6) Variants

## 6.1 Tokens per step and confidence-threshold decoding

The same MDLM checkpoint, with more tokens committed per pass (fixed $k$), or with every token
whose confidence exceeds $\tau = 0.9$ (Fast-dLLM's parallel decoding).

In [ ]:
variants = [
    ("MDLM, 4 tokens/step", {"tokens_per_step": 4}),
    ("MDLM, confidence > 0.9", {"threshold": 0.9}),
    ("BD3-LM + KV cache, confidence > 0.9", {"threshold": 0.9}),
]
for name, kwargs in variants:
    generate = (
        block_diffusion_generate if name.startswith("BD3") else diffusion_generate
    )
    chosen = block_model if name.startswith("BD3") else model
    results.append(evaluate(name, lambda q: generate(chosen, q, **kwargs)))
print_results(results)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for r in results:
    ax.scatter(r["tokens_per_pass"], r["accuracy"], s=30)
    ax.annotate(
        r["method"],
        (r["tokens_per_pass"], r["accuracy"]),
        fontsize=7,
        xytext=(4, 2),
        textcoords="offset points",
    )
ax.set_xlabel("reply tokens per forward pass")
ax.set_ylabel(f"GSM8K accuracy ({NUM_QUESTIONS} questions)")
plt.tight_layout()
plt.show()

Fixed $k$ trades accuracy for passes: every step commits $k$ tokens whether or not the model is
sure of them, and conflicting commits of dependent tokens compound across the reasoning chain.
The threshold rule adapts per step: it commits many tokens where the text is predictable
(formatting, copied numbers, padding) and one where it is not, so it keeps more accuracy for its
speed-up. Combined with BD3-LM's exact cache, it needs less than half the passes of the
one-token run and answers faster than the AR model, on the same 0.6B weights. On ten questions,
treat the accuracy column as a direction, not a measurement.

## 6.2 Infilling

A bidirectional model can condition on text to the *right* of the gap, which a left-to-right
model cannot do without special training. Here the reply's last sentence is given and the
reasoning before it is masked. The model has to write a derivation that ends at the given answer.

In [ ]:
@torch.no_grad()
def infill(
    model: Qwen3ForCausalLM, prompt: str, suffix: str, num_masks: int, **commit_kwargs
) -> str:
    """Fill num_masks masks between prompt and suffix in one block -> middle text."""
    x = masked_canvas(prompt, num_masks)
    suffix_ids = tokenizer(suffix, return_tensors="pt").input_ids.to(DEVICE)
    x = torch.cat([x, suffix_ids], dim=1)
    gap = slice(
        x.shape[1] - suffix_ids.shape[1] - num_masks, x.shape[1] - suffix_ids.shape[1]
    )
    while (x[0, gap] == MASK_ID).any():
        logits = model(x, attention_mask=full_mask(x.shape[1])).logits[0, gap]
        x[0, gap] = commit_tokens(logits, x[0, gap], **commit_kwargs)
    return tokenizer.decode(x[0, gap])


prompt = chat_prompt(gsm_test[1]["question"])
gold = gsm_test[1]["answer"].split("####")[-1].strip()
print(gsm_test[1]["question"], "\n")
print(
    infill(model, prompt, f" So the answer is \\boxed{{{gold}}}.<|im_end|>", 64), end=""
)
print(f" So the answer is \\boxed{{{gold}}}.")

---

# 7) Practical notes

**Where the speed comes from.** A dLLM pass costs about as much as an AR prefill of the same
length. The win requires (1) many tokens per pass, (2) a KV cache so each pass does not re-read
the whole context, and (3) a batch small enough that AR decoding is bandwidth bound. Commercial
dLLMs report >1,000 tokens/s by combining all three with distillation to few steps. At large
batch sizes AR decoding is compute bound too, and the advantage shrinks.

**Caching.** Full-sequence MDLM has no exact cache. Fast-dLLM reuses stale prefix and suffix
keys for a few steps and refreshes them per block, which works because keys of committed tokens
change little. Block diffusion (BD3-LM, LLaDA 2.x, SDAR) gets an exact cache by construction, and
is the direction recent large dLLMs have taken.

**Length.** MDLM needs `gen_len` in advance and pays for masks it never uses. Short replies get
padded with end-of-sequence tokens, and too short a canvas truncates the answer. Blocks fix both.

**Hyperparameters.** Block size 32–64; a confidence threshold of 0.8–0.95; greedy or
low-temperature decoding for reasoning. LLaDA-style models also benefit from classifier-free
guidance (`cfg_scale`), which doubles the passes.

**Failure modes.** Repetition when many neighbouring positions commit in one step; early end of
sequence when padding is committed before the content; reasoning chains that commit the answer
before the derivation (visible in the decoding-order plot), so the derivation is written to fit
the answer.

**Training at scale.** Adapting an AR checkpoint (Dream, the dLLM recipes) is far cheaper than
training from scratch (LLaDA). It keeps the AR model's knowledge, but costs some quality at small
scale (GSM8K: 59.6 for the AR base model vs 46.6 for BD3-LM and 29.8 for MDLM). RL for dLLMs (diffu-GRPO, VRPO in LLaDA 1.5) has to
estimate sequence log-likelihoods through the ELBO, since there is no exact likelihood.

**Where dLLMs win today.** Code infilling and editing, latency-bound single-user serving, and
tasks with global constraints (Sudoku-like planning, filling a template). AR still wins on
quality per parameter and on mature serving stacks.

---

# 8) Conclusion

| Need | Choice | Why |
|---|---|---|
| Best quality per parameter, mature serving | AR | Exact likelihood, KV cache, one order that matches language |
| Lowest latency at small batch | Block diffusion + confidence-threshold decoding | Several tokens per pass with an exact cache |
| Fill a gap given text on both sides | Masked diffusion (MDLM / LLaDA) | Bidirectional conditioning is native |
| Quality-latency knob at inference | Any dLLM | Tokens per pass or the threshold sets the trade-off, no retraining |
| Training a new dLLM | Adapt an AR checkpoint with the masked-diffusion ELBO | Inherits AR knowledge; a weighted masked-LM loss is all that changes |